# 03 · Motor 2 (Despachos) + Motor 3 (Producción)

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','pandas','numpy','plotly','python-dotenv','supabase','openpyxl'])
print('OK')

In [ ]:
import pandas as pd, numpy as np, ast, os, sys, warnings
from pathlib import Path
from datetime import date, timedelta
import plotly.express as px
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))
from conexion import get_supabase_client, supabase_to_df, df_to_supabase
sb = get_supabase_client()
COBERTURA_MIN=2.5; COBERTURA_OBJ=6.0; COBERTURA_MAX=8.0
GMROII_MIN=1.8; SEMANAS_PROD=16; STOCK_SEG=0.15
print('Config OK')

## Motor 2 · Despachos

In [ ]:
df_inv   = supabase_to_df(sb,'fact_inventario_semanal')
df_tipos = supabase_to_df(sb,'dim_tipos_producto')
df_tiend = supabase_to_df(sb,'dim_tiendas')
df_fc4   = supabase_to_df(sb,'output_forecast_semanal')
df_inv['tienda_id']=df_inv['tienda_id'].astype(int)
df_tiend['tienda_id']=df_tiend['tienda_id'].astype(int)
# Ultimo inventario
fecha_max=df_inv['fecha'].max()
df_inv=df_inv[df_inv['fecha']==fecha_max].merge(df_tiend[['tienda_id','nombre_tienda','ciudad','formato']],on='tienda_id',how='left')
# Forecast 4 semanas
df_fc4=df_fc4.groupby(['tienda_id','tipo_producto','familia']).agg(forecast_4sem=('forecast_medio','sum')).reset_index()
df_fc4['tienda_id']=df_fc4['tienda_id'].astype(int)
print(f'Inventario:{len(df_inv):,} | Forecast:{len(df_fc4):,}')

In [ ]:
precio_d={r['tipo_producto']:r['precio_regular'] for _,r in df_tipos.iterrows()}
costo_d ={r['tipo_producto']:r['costo_produccion'] for _,r in df_tipos.iterrows()}
margen_d={k:(precio_d[k]-costo_d[k])/precio_d[k] for k in precio_d}
def gmroii(tipo,u_desp,fc4):
    if tipo not in precio_d or u_desp<=0: return 0
    m=min(u_desp,fc4)*precio_d[tipo]*margen_d[tipo]
    c=u_desp*costo_d[tipo]
    return round(m/c,2) if c>0 else 0
df_m=df_inv.merge(df_fc4[['tienda_id','tipo_producto','forecast_4sem']],on=['tienda_id','tipo_producto'],how='left').fillna({'forecast_4sem':0})
df_m['venta_sem']=df_m['forecast_4sem']/4
df_m['cobertura']= ((df_m['unidades_disponibles']+df_m['unidades_transito'].fillna(0))/df_m['venta_sem'].replace(0,np.nan)).fillna(99).round(1)
df_m['u_sugeridas']=np.where(df_m['cobertura']<COBERTURA_MIN,(COBERTURA_OBJ*df_m['venta_sem']-df_m['unidades_disponibles']).clip(lower=0).round(0).astype(int),0)
desp=df_m[df_m['u_sugeridas']>0].copy()
desp['gmroii']=desp.apply(lambda r:gmroii(r['tipo_producto'],r['u_sugeridas'],r['forecast_4sem']),axis=1)
desp['tipo_despacho']='RESURTIDO'
desp['estado']=np.where(desp['gmroii']>=GMROII_MIN,'APROBADO','REVISAR')
desp['fecha_ejecucion']=str(date.today())
desp['semana_despacho']=str(date.today()+timedelta(days=7))
print(f'Despachos:{len(desp):,} | Aprobados:{(desp.estado=="APROBADO").sum():,} | Revisar:{(desp.estado=="REVISAR").sum():,}')

In [ ]:
cols=['fecha_ejecucion','tienda_id','tipo_producto','tipo_despacho','u_sugeridas','semana_despacho','cobertura','gmroii','estado']
df_save=desp[cols].rename(columns={'u_sugeridas':'unidades_sugeridas','cobertura':'cobertura_actual','gmroii':'gmroii_proyectado'})
df_to_supabase(sb,df_save,'output_despachos_recomendados',limpiar_hoy=True)
os.makedirs('../outputs',exist_ok=True)
desp.to_excel('../outputs/despachos_semana.xlsx',index=False)
print('Excel: outputs/despachos_semana.xlsx')

In [ ]:
heat=df_m.groupby(['tipo_producto','ciudad'])['cobertura'].mean().reset_index()
pivot=heat.pivot(index='tipo_producto',columns='ciudad',values='cobertura')
px.imshow(pivot.round(1),title='Cobertura inventario (semanas)',
    color_continuous_scale=[[0,'#d32f2f'],[0.2,'#ff9800'],[0.45,'#4caf50'],[0.7,'#ffeb3b'],[1,'#f44336']],
    zmin=0,zmax=12,text_auto=True,aspect='auto').show()

## Motor 3 · Producción

In [ ]:
df_fc_all=supabase_to_df(sb,'output_forecast_semanal')
dem=df_fc_all.groupby(['tipo_producto','familia']).agg(demanda_8sem=('forecast_medio','sum')).reset_index()
inv_red=df_m.groupby('tipo_producto').agg(inv_red=('unidades_disponibles','sum')).reset_index()
plan=dem.merge(inv_red,on='tipo_producto',how='left').fillna({'inv_red':0})
plan=plan.merge(df_tipos[['tipo_producto','costo_produccion','tallas_json']],on='tipo_producto',how='left')
plan['stock_seg']=(plan['demanda_8sem']*STOCK_SEG).round(0)
plan['u_producir']=(plan['demanda_8sem']+plan['stock_seg']-plan['inv_red']).clip(lower=0).round(0).astype(int)
hoy=date.today()
prod_rows=[]
for _,row in plan.iterrows():
    try: tallas=ast.literal_eval(str(row['tallas_json']))
    except: tallas={'S':0.25,'M':0.35,'L':0.25,'XL':0.15}
    total=row['u_producir']
    dist={t:int(total*p) for t,p in tallas.items()}
    prod_rows.append({'fecha_ejecucion':str(hoy),'tipo_producto':row['tipo_producto'],
        'familia':row['familia'],'semana_inicio_prod':str(hoy+timedelta(weeks=1)),
        'semana_llegada_cedi':str(hoy+timedelta(weeks=SEMANAS_PROD)),
        'unidades_totales':total,'distribucion_tallas':str(dist),
        'inversion_estimada':int(total*row['costo_produccion']),'zona_pipeline':'AZUL','estado':'RECOMENDADO'})
df_prod=pd.DataFrame(prod_rows)
df_to_supabase(sb,df_prod,'output_produccion_recomendada',limpiar_hoy=True)
df_prod.to_excel('../outputs/produccion_recomendada.xlsx',index=False)
print('Excel: outputs/produccion_recomendada.xlsx')

In [ ]:
px.bar(df_prod[df_prod['unidades_totales']>0].sort_values('inversion_estimada'),
    x='inversion_estimada',y='tipo_producto',orientation='h',color='familia',
    title='Inversion estimada produccion').show()
print('=== RESUMEN EJECUTIVO ===')
print(f'Motor 2 - Despachos: {len(desp):,} | Aprobados: {(desp.estado=="APROBADO").sum():,}')
print(f'Motor 3 - Produccion: {len(df_prod):,} tipos | ${df_prod.inversion_estimada.sum():,.0f} COP')
print(f'Llegada CEDI: {hoy+timedelta(weeks=SEMANAS_PROD)}')
print('OK - Proyecto completo')